In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from pathlib import Path

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
tfm = transforms.ToTensor()
full_train = datasets.MNIST("data", train=True,  download=True, transform=tfm)
test_ds    = datasets.MNIST("data", train=False, download=True, transform=tfm)

train_ds, val_ds = random_split(full_train, [55_000, 5_000],
                                generator=torch.Generator().manual_seed(42))

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=128)
test_dl  = DataLoader(test_ds,  batch_size=16)   # small batch for image preview

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder:  1×28×28  ->  latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # 16×14×14
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 32×7×7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim),
        )
        # Decoder:  latent_dim  ->  1×28×28
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2,
                               padding=1, output_padding=1),       # 16×14×14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2,
                               padding=1, output_padding=1),       # 1×28×28
            nn.Sigmoid(),   # pixel values back into [0, 1]
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z      # reconstruction, latent code

In [ ]:
model     = ConvAutoencoder(latent_dim=32).to(device)
loss_fn   = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

log_dir = Path("/content/runs/autoencoder").resolve()
writer  = SummaryWriter(log_dir)
print(f"Logging to {log_dir}")

# grab a fixed batch we'll re-encode every epoch, so we can watch progress
fixed_imgs, fixed_labels = next(iter(test_dl))
fixed_imgs = fixed_imgs.to(device)

# log the originals once, as a reference row
writer.add_image("reconstructions/original",
                 make_grid(fixed_imgs.cpu(), nrow=8), global_step=0)

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, _ in loader:
            x = x.to(device)
            x_hat, _ = model(x)
            loss = loss_fn(x_hat, x)        # reconstruct the input
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            n          += x.size(0)
    return total_loss / n

In [ ]:
EPOCHS = 10
best_val = float("inf")